### Kết Luận Chính Từ Dữ Liệu Tổng Thể

1. **Phân khúc giá trọng tâm**: Hơn 65% giao dịch xe cũ tập trung trong khoảng giá **250 - 650 triệu VNĐ**. Phân khúc xe dưới 300 triệu chủ yếu là xe hạng A (Morning, i10) hoặc xe trên 10 năm tuổi.
2. **Tốc độ khấu hao theo năm**: Xe mất giá nhanh nhất trong 3–5 năm đầu (khoảng 8–10%/năm). Sau mốc 7–10 năm, đường cong khấu hao thoải dần và tiệm cận mức sàn 150–250 triệu VNĐ đối với xe phổ thông.
3. **Tương quan giữa ODO và Giá bán**: Năm sản xuất (đời xe) chi phối giá mạnh hơn ODO. Một chiếc xe đời 2022 chạy 80.000 km vẫn có giá thị trường cao hơn đáng kể so với cùng dòng xe đời 2017 chạy 40.000 km.


In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path
# Đọc dữ liệu sạch
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
CLEAN_PATH = ROOT / "data" / "processed" / "cars_cleaned.csv"
df = pd.read_csv(CLEAN_PATH, low_memory=False)


In [2]:
# 1. Khởi tạo danh sách tùy chọn cho bộ lọc
brands = ['Tất cả'] + sorted(df['carbrand_name'].dropna().unique().tolist())
provinces = ['Tất cả'] + sorted(df['region_name'].dropna().unique().tolist())
min_year = int(df['mfdate'].min())
max_year = int(df['mfdate'].max())

In [3]:
# 2. Khởi tạo các Widgets điều khiển
w_brand = widgets.Dropdown(
    options=brands,
    value='Tất cả',
    description='Hãng xe:',
    style={'description_width': 'initial'}
)
w_model = widgets.Dropdown(
    options=['Tất cả'],
    value='Tất cả',
    description='Dòng xe:',
    style={'description_width': 'initial'}
)
w_province = widgets.Dropdown(
    options=provinces,
    value='Tất cả',
    description='Tỉnh/Thành:',
    style={'description_width': 'initial'}
)
w_year = widgets.IntRangeSlider(
    value=[max(2012, min_year), max_year],
    min=min_year,
    max=max_year,
    step=1,
    description='Khoảng năm:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='380px')
)

In [4]:
# 3. Liên kết phụ thuộc: Đổi Hãng xe thì cập nhật danh sách Dòng xe
def on_brand_change(*args):
    if w_brand.value == 'Tất cả':
        w_model.options = ['Tất cả']
    else:
        models = sorted(df[df['carbrand_name'] == w_brand.value]['carmodel_name'].dropna().unique().tolist())
        w_model.options = ['Tất cả'] + models
    w_model.value = 'Tất cả'
w_brand.observe(on_brand_change, 'value')


In [5]:
# 4. Vùng hiển thị biểu đồ và hàm cập nhật dữ liệu
out_dashboard = widgets.Output()
def update_dashboard(*args):
    with out_dashboard:
        clear_output(wait=True)
        
        # Lọc dữ liệu theo giá trị widget
        sub_df = df.copy()
        if w_brand.value != 'Tất cả':
            sub_df = sub_df[sub_df['carbrand_name'] == w_brand.value]
        if w_model.value != 'Tất cả':
            sub_df = sub_df[sub_df['carmodel_name'] == w_model.value]
        if w_province.value != 'Tất cả':
            sub_df = sub_df[sub_df['region_name'] == w_province.value]
        
        sub_df = sub_df[(sub_df['mfdate'] >= w_year.value[0]) & (sub_df['mfdate'] <= w_year.value[1])]
        
        # Kiểm tra tập dữ liệu rỗng
        if len(sub_df) == 0:
            print("Không tìm thấy dữ liệu phù hợp với bộ lọc hiện tại. Vui lòng mở rộng khoảng năm hoặc chọn tỉnh/thành khác.")
            return
        # Hiển thị chỉ số nhanh (KPI summary)
        n_cars = len(sub_df)
        mean_price = sub_df['price_million'].mean()
        median_price = sub_df['price_million'].median()
        median_odo = sub_df['mileage_v2'].median()
        
        print(f"Số lượng tin: {n_cars:,} xe | Giá trung bình: {mean_price:.1f} triệu | Giá trung vị: {median_price:.1f} triệu | ODO trung vị: {median_odo:,.0f} km")
        # Biểu đồ 1: Phân phối giá bán kèm Boxplot
        fig1 = px.histogram(
            sub_df,
            x='price_million',
            nbins=35,
            marginal='box',
            color_discrete_sequence=['#1f77b4'],
            title=f"Phân Phối Giá Xe (Trung vị: {median_price:.1f} Triệu VNĐ)",
            labels={'price_million': 'Giá bán (Triệu VNĐ)'}
        )
        fig1.update_layout(
            bargap=0.08,
            yaxis_title="Số lượng xe",
            margin=dict(l=40, r=20, t=50, b=40),
            height=380
        )
        fig1.show()
        # Biểu đồ 2: Tương quan ODO vs Giá bán theo Năm sản xuất
        fig2 = px.scatter(
            sub_df,
            x='mileage_v2',
            y='price_million',
            color='mfdate',
            color_continuous_scale='Turbo',
            title="Tương Quan ODO - Giá Bán - Năm Sản Xuất",
            labels={
                'mileage_v2': 'Số Km đã chạy (ODO)',
                'price_million': 'Giá bán (Triệu VNĐ)',
                'mfdate': 'Năm SX'
            },
            hover_data=['carmodel_name', 'region_name']
        )
        fig2.update_layout(
            margin=dict(l=40, r=20, t=50, b=40),
            height=420
        )
        fig2.show()

In [6]:
# 5. Gắn sự kiện theo dõi thay đổi vào các Widgets
w_brand.observe(update_dashboard, 'value')
w_model.observe(update_dashboard, 'value')
w_province.observe(update_dashboard, 'value')
w_year.observe(update_dashboard, 'value')


In [7]:
# 6. Sắp xếp bố cục giao diện
dashboard_ui = widgets.VBox([
    widgets.HBox([w_brand, w_model]),
    widgets.HBox([w_province, w_year]),
    widgets.HTML("<hr style='margin: 10px 0;'>"),
    out_dashboard
])
display(dashboard_ui)
# Khởi chạy lần đầu để nạp biểu đồ mặc định
update_dashboard()
